# Global AI Dataset Cleaning



# 1. Import Libraries and Load Raw Data



In [1]:
# Import pandas for data cleaning and table operations

import pandas as pd

# Load raw CSV files

respondents = pd.read_csv('../data/raw/respondents.csv')
usage = pd.read_csv('../data/raw/ai_tool_usage.csv')
salary = pd.read_csv('../data/raw/salary_and_satisfaction.csv')
skills = pd.read_csv('../data/raw/skills_and_training.csv')
ethics = pd.read_csv('../data/raw/ai_policy_and_ethics.csv')

# 2. Convert Respondent Variables

Categorical variables in the respondents table are converted to the `category` data type.

This improves readability and makes the intended variable type clearer for later analysis.

In [2]:
# Convert respondent-level categorical variables to category dtype

cat_cols = [
    "gender",
    "country",
    "education_level",
    "job_role",
    "industry",
    "company_size",
    "remote_work_status"
]

for col in cat_cols:
    respondents[col] = respondents[col].astype("category")

In [3]:
# Convert respondent-level numerical variables to appropriate numeric types

respondents["age"] = pd.to_numeric(
    respondents["age"],
    downcast="integer"
)

respondents["years_of_experience"] = pd.to_numeric(
    respondents["years_of_experience"],
    downcast="integer"
)

respondents["survey_year"] = pd.to_numeric(
    respondents["survey_year"],
    downcast="integer"
)

In [4]:
# Convert survey year to category because it represents a survey period

respondents["survey_year"] = respondents["survey_year"].astype("category")

# 3. Check Respondent ID Consistency

Before merging confirm how many unique `respondent_id` values are present in each table.

This check helps identify whether a table contains one row per respondent or multiple rows per respondent.

In [5]:
# Compare the number of unique respondent IDs across all respondent-level tables

for name, table in {
    "respondents": respondents,
    "salary": salary,
    "skills": skills,
    "ethics": ethics
}.items():
    print(
        f"{name}:",
        "shape =", table.shape,
        "| unique respondent_id =", table["respondent_id"].nunique()
    )

respondents: shape = (12000, 11) | unique respondent_id = 12000
salary: shape = (12000, 10) | unique respondent_id = 12000
skills: shape = (12000, 8) | unique respondent_id = 12000
ethics: shape = (12000, 7) | unique respondent_id = 12000


### ID consistency interpretation

The respondents, salary, skills, and ethics tables are expected to contain one row per respondent.

The AI usage table is handled separately because one respondent can report several AI tools. Therefore, the usage table needs to be aggregated before it can be merged into the master dataset.

# 4. Convert Skills and Salary Variables

Numerical variables in the skills and salary tables are converted to numeric types.

This prepares the variables for summary statistics, visualizations, and later modeling.

In [6]:
# Convert numerical variables in the skills table

skills["num_skills"] = pd.to_numeric(
    skills["num_skills"],
    downcast="integer"
)

skills["weekly_learning_hours"] = pd.to_numeric(
    skills["weekly_learning_hours"],
    downcast="integer"
)

In [7]:
# Convert numerical variables in the salary and satisfaction table

salary["annual_salary_usd"] = pd.to_numeric(
    salary["annual_salary_usd"],
    downcast="integer"
)

salary["salary_change_yoy_pct"] = pd.to_numeric(
    salary["salary_change_yoy_pct"],
    downcast="integer"
)

salary["job_satisfaction_score"] = pd.to_numeric(
    salary["job_satisfaction_score"],
    downcast="integer"
)

salary["work_life_balance_score"] = pd.to_numeric(
    salary["work_life_balance_score"],
    downcast="integer"
)

# 5. Prepare AI Tool Usage Data

The AI usage table contains information about individual AI tools used by respondents.

Because one respondent may use multiple tools, this table must be aggregated to respondent level before merging.

In [8]:
# Preview the AI usage table

usage.head()

,respondent_id,ai_tool,usage_frequency,productivity_change_pct,satisfaction_score,primary_use_case
0,R00001,v0 by Vercel,Monthly,39.8,3,Data analysis
1,R00001,Stable Diffusion,Several times a week,21.9,5,Customer support automation
2,R00001,LLaMA / Open-source LLMs,Weekly,-1.0,4,Data analysis
3,R00001,Replit AI,Monthly,28.8,3,Image generation
4,R00002,Cursor,Monthly,49.7,5,Image generation


In [9]:
# Check the distribution of AI usage frequency categories

usage["usage_frequency"].value_counts()

usage_frequency
Several times a week    14032
Daily                   11729
Weekly                   9492
Monthly                  7208
Rarely                   4745
Name: count, dtype: int64

## Convert Usage Frequency to Numerical Scale

The original usage frequency variable is categorical. To calculate average usage frequency per respondent, it is converted to an ordinal numerical scale:

- Rarely = 1
- Monthly = 2
- Weekly = 3
- Several times a week = 4
- Daily = 5

In [10]:
# Define ordinal mapping for AI usage frequency

freq_map = {
    "Rarely": 1,
    "Monthly": 2,
    "Weekly": 3,
    "Several times a week": 4,
    "Daily": 5
}

In [11]:
# Apply the frequency mapping to create a numerical usage variable

usage["usage_frequency_num"] = (
    usage["usage_frequency"]
    .map(freq_map)
)

In [12]:
# Verify that the original and numerical frequency variables match

usage[
    ["usage_frequency", "usage_frequency_num"]
].head()

,usage_frequency,usage_frequency_num
0,Monthly,2
1,Several times a week,4
2,Weekly,3
3,Monthly,2
4,Monthly,2


## Aggregate AI Usage to Respondent Level

The usage table is aggregated by `respondent_id`.

For each respondent, the aggregation creates:

- number of different AI tools used,
- average AI usage frequency,
- average productivity change,
- average tool satisfaction.

In [13]:
# Aggregate the AI usage table to one row per respondent

usage_summary = (
    usage
    .groupby("respondent_id")
    .agg(
        num_ai_tools=("ai_tool", "nunique"),
        avg_usage_frequency=("usage_frequency_num", "mean"),
        avg_productivity_change_pct=("productivity_change_pct", "mean"),
        avg_tool_satisfaction=("satisfaction_score", "mean")
    )
    .reset_index()
)

In [14]:
# Check the aggregated AI usage table

print("Shape:", usage_summary.shape)
usage_summary.head()

Shape: (12000, 5)


,respondent_id,num_ai_tools,avg_usage_frequency,avg_productivity_change_pct,avg_tool_satisfaction
0,R00001,4,2.75,22.375,3.75
1,R00002,5,3.20,24.060,4.00
2,R00003,1,4.00,45.300,4.00
3,R00004,4,4.50,21.200,4.00
4,R00005,2,4.50,37.600,3.50


### Usage aggregation interpretation

The AI usage table originally contained multiple records per respondent because each respondent could report several AI tools.

After aggregation, the table contains one row per respondent and can be safely merged with the other respondent-level tables.

# 6. Merge Tables into Master Dataset

All respondent-level tables are merged using `respondent_id`.

The final dataset combines demographic, job-related, salary, skills, ethics, and AI usage information.

In [15]:
# Merge all cleaned respondent-level tables into one master DataFrame

master_df = (
    respondents
    .merge(salary, on="respondent_id")
    .merge(skills, on="respondent_id")
    .merge(ethics, on="respondent_id")
    .merge(usage_summary, on="respondent_id")
)

In [16]:
# Check the shape of the merged master dataset

master_df.shape

(12000, 37)

In [17]:
# Review the structure of the merged master dataset

master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 37 columns):
 #   Column                          Non-Null Count  Dtype   
---  ------                          --------------  -----   
 0   respondent_id                   12000 non-null  str     
 1   survey_year                     12000 non-null  category
 2   age                             12000 non-null  int8    
 3   gender                          12000 non-null  category
 4   country                         12000 non-null  category
 5   education_level                 12000 non-null  category
 6   job_role                        12000 non-null  category
 7   industry                        12000 non-null  category
 8   company_size                    12000 non-null  category
 9   years_of_experience             12000 non-null  int8    
 10  remote_work_status              12000 non-null  category
 11  annual_salary_usd               12000 non-null  int32   
 12  salary_change_yoy_pct        

# 7. Convert Master Dataset Variable Types

After merging, additional categorical and numerical variables are converted to appropriate types.



In [18]:
# Convert additional categorical variables to category dtype

cat_cols = [
    "career_growth_outlook",
    "ai_job_replacement_fear",
    "job_search_status",
    "received_ai_related_promotion",
    "primary_learning_platform",
    "learned_genai_skills_2024_2026",
    "employer_provides_ai_training",
    "company_has_ai_policy",
    "supports_ai_regulation",
    "has_experienced_ai_misuse",
    "trust_in_ai_decisions"
]

for col in cat_cols:
    master_df[col] = master_df[col].astype("category")

In [19]:
# Convert integer-like numerical variables in the master dataset

for col in [
    "ai_bias_concern_level",
    "data_privacy_concern_level",
    "num_ai_tools"
]:
    master_df[col] = pd.to_numeric(
        master_df[col],
        downcast="integer"
    )

In [20]:
# Convert continuous numerical variables in the master dataset

for col in [
    "salary_change_yoy_pct",
    "avg_usage_frequency",
    "avg_productivity_change_pct",
    "avg_tool_satisfaction"
]:
    master_df[col] = pd.to_numeric(
        master_df[col],
        downcast="float"
    )

In [21]:
# Review final data types after conversion

master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 37 columns):
 #   Column                          Non-Null Count  Dtype   
---  ------                          --------------  -----   
 0   respondent_id                   12000 non-null  str     
 1   survey_year                     12000 non-null  category
 2   age                             12000 non-null  int8    
 3   gender                          12000 non-null  category
 4   country                         12000 non-null  category
 5   education_level                 12000 non-null  category
 6   job_role                        12000 non-null  category
 7   industry                        12000 non-null  category
 8   company_size                    12000 non-null  category
 9   years_of_experience             12000 non-null  int8    
 10  remote_work_status              12000 non-null  category
 11  annual_salary_usd               12000 non-null  int32   
 12  salary_change_yoy_pct        

# 8. Handle Meaningful Empty Categories

Some empty values in the raw data are not treated as missing information but as meaningful categories.

Two variables require special handling:

- `burnout_level`
- `top_certification`

## Burnout Level

In the raw data, empty burnout values are interpreted as respondents reporting no burnout symptoms.

Therefore, empty values are replaced with `No burnout`.

In [22]:
# Check burnout level distribution before recoding

master_df["burnout_level"].value_counts(dropna=False)

burnout_level
Mild        4158
Moderate    3703
NaN         2414
Severe      1725
Name: count, dtype: int64

In [23]:
# Load salary data again without automatic NA conversion to inspect raw burnout values

burnout_raw = pd.read_csv(
    "../data/raw/salary_and_satisfaction.csv",
    keep_default_na=False
)

burnout_raw["burnout_level"].value_counts(dropna=False)

burnout_level
Mild        4158
Moderate    3703
None        2414
Severe      1725
Name: count, dtype: int64

In [24]:
# Replace missing burnout values with a meaningful category

master_df["burnout_level"] = (
    master_df["burnout_level"]
    .astype("string")
    .fillna("No burnout")
    .astype("category")
)

In [25]:
# Verify burnout level distribution after recoding

master_df["burnout_level"].value_counts()

burnout_level
Mild          4158
Moderate      3703
No burnout    2414
Severe        1725
Name: count, dtype: int64

## Certification Information

Missing values in `top_certification` are interpreted as respondents having no certification.

These values are replaced with `No Certification`.

In [26]:
# Check certification distribution before recoding

master_df["top_certification"].value_counts(dropna=False)

top_certification
NaN                                 3630
DeepLearning.AI Specialization      1271
Kaggle Competitions Expert          1220
Coursera ML Certificate             1219
Azure AI Engineer                   1168
Google Professional ML Engineer     1167
AWS ML Specialty                    1163
TensorFlow Developer Certificate    1162
Name: count, dtype: int64

In [27]:
# Replace missing certification values with a meaningful category

master_df["top_certification"] = (
    master_df["top_certification"]
    .fillna("No Certification")
)

In [28]:
# Convert certification variable to category dtype

master_df["top_certification"] = (
    master_df["top_certification"]
    .astype("category")
)

In [29]:
# Verify certification distribution after recoding

master_df["top_certification"].value_counts(dropna=False)

top_certification
No Certification                    3630
DeepLearning.AI Specialization      1271
Kaggle Competitions Expert          1220
Coursera ML Certificate             1219
Azure AI Engineer                   1168
Google Professional ML Engineer     1167
AWS ML Specialty                    1163
TensorFlow Developer Certificate    1162
Name: count, dtype: int64

# 9. Final Quality Check

Before exporting the dataset, the final structure and missing values are checked one more time.

In [30]:
# Check whether any missing values remain in the master dataset

missing_final = master_df.isna().sum().sort_values(ascending=False)
missing_final[missing_final > 0]

Series([], dtype: int64)

In [31]:
# Display final dataset dimensions

master_df.shape

(12000, 37)

In [32]:
# Preview the cleaned master dataset

master_df.head()

,respondent_id,survey_year,age,gender,country,education_level,job_role,industry,company_size,years_of_experience,...,company_has_ai_policy,ai_bias_concern_level,data_privacy_concern_level,supports_ai_regulation,has_experienced_ai_misuse,trust_in_ai_decisions,num_ai_tools,avg_usage_frequency,avg_productivity_change_pct,avg_tool_satisfaction
0,R00001,2025,30,Male,Nigeria,PhD,Data Analyst,E-commerce,1001-5000,10,...,No,5,5,Neutral,No,Low,4,2.75,22.375000,3.75
1,R00002,2025,36,Female,Spain,Master's,Data Analyst,Legal,5000+,6,...,Don't know,3,1,Agree,No,High,5,3.20,24.059999,4.00
2,R00003,2025,18,Male,Spain,High School,AI Ethics / Policy,Technology,5000+,11,...,Yes - formal,5,5,Strongly agree,Unsure,Moderate,1,4.00,45.299999,4.00
3,R00004,2024,31,Female,India,Bachelor's,Full-Stack Developer,Telecom,5000+,21,...,No,3,4,Agree,No,Low,4,4.50,21.200001,4.00
4,R00005,2026,34,Female,Nigeria,Master's,Data Scientist,Media & Entertainment,1-10,6,...,Yes - informal,3,3,Agree,Unsure,High,2,4.50,37.599998,3.50


# 10. Export Cleaned Master Dataset

The cleaned and merged dataset is exported to the `data/processed` folder.

This file will be used in the next notebook for exploratory data analysis.

In [33]:
# Export the cleaned master dataset

master_df.to_csv(
    "../data/processed/global_ai_master.csv",
    index=False
)

# Cleaning Summary

The data cleaning process included the following steps:

- loaded five raw CSV files,
- converted categorical and numerical variables to appropriate data types,
- checked respondent ID consistency,
- converted AI usage frequency to an ordinal numerical scale,
- aggregated AI usage data to respondent level,
- merged all respondent-level tables into one master dataset,
- recoded empty burnout values as `No burnout`,
- recoded missing certification values as `No Certification`,
- exported the final cleaned dataset.

The resulting file `global_ai_master.csv` is ready for EDA.